# JOTATIGER.FIT — Servidor de Voz (XTTS v2)

Genera audio con voz masculina realista en español para Juan Fit.

**Pasos:**
1. Ejecuta todas las celdas en orden
2. Copia la URL de ngrok que aparece al final
3. Pégala en tu `.env` como `COLAB_WEBHOOK_URL`

⚠️ Asegúrate de tener **GPU T4** activada: Entorno de ejecución → Cambiar tipo → T4

In [ ]:
# CELDA 1 — Instalar dependencias
!pip install TTS flask flask-ngrok pyngrok numpy==1.26.4 -q
!pip install pyngrok --upgrade -q

In [ ]:
# CELDA 2 — Configurar ngrok
# Consigue tu token gratis en: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "PON_TU_TOKEN_NGROK_AQUI"  # <-- cambia esto

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)

In [ ]:
# CELDA 3 — Cargar modelo XTTS v2
import torch
from TTS.api import TTS

print('GPU disponible:', torch.cuda.is_available())
print('Cargando modelo XTTS v2... (puede tardar 2-3 min la primera vez)')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('Modelo cargado correctamente')

In [ ]:
# CELDA 4 — Crear audio de referencia para la voz de Juan Fit
# Esto genera un audio base con voz masculina estándar
# Si tienes un archivo .wav de referencia, súbelo y cambia la ruta

import os

REFERENCE_WAV = '/content/juan_fit_reference.wav'

if not os.path.exists(REFERENCE_WAV):
    # Generar audio de referencia con voz por defecto
    tts.tts_to_file(
        text="Hola, soy Juan Fit. Sin rollos, aquí va la verdad sobre el fitness y la salud.",
        speaker=tts.speakers[0] if tts.speakers else None,
        language='es',
        file_path=REFERENCE_WAV
    )
    print('Audio de referencia creado:', REFERENCE_WAV)
else:
    print('Usando audio de referencia existente:', REFERENCE_WAV)

In [ ]:
# CELDA 5 — Servidor Flask + ngrok
import threading
import uuid
from flask import Flask, request, jsonify, send_file
from pyngrok import ngrok

app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'xtts_v2', 'device': device})

@app.route('/tts', methods=['POST'])
def text_to_speech():
    try:
        data = request.get_json()
        text = data.get('text', '')
        language = data.get('language', 'es')
        speaker_wav = data.get('speaker_wav', REFERENCE_WAV)

        if not text:
            return jsonify({'error': 'texto vacío'}), 400

        output_path = f'/content/audio_{uuid.uuid4().hex[:8]}.wav'

        tts.tts_to_file(
            text=text,
            speaker_wav=speaker_wav,
            language=language,
            file_path=output_path
        )

        return send_file(output_path, mimetype='audio/wav')

    except Exception as e:
        return jsonify({'error': str(e)}), 500

# Iniciar servidor en hilo separado
def run_server():
    app.run(port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_server)
thread.daemon = True
thread.start()

# Exponer con ngrok
import time
time.sleep(2)
tunnel = ngrok.connect(5000)

print('=' * 50)
print('SERVIDOR DE VOZ ACTIVO')
print('=' * 50)
print(f'URL pública: {tunnel.public_url}')
print(f'Endpoint TTS: {tunnel.public_url}/tts')
print(f'Health check: {tunnel.public_url}/health')
print('=' * 50)
print('Copia esta URL en tu .env como COLAB_WEBHOOK_URL')

## Test manual
Ejecuta la celda de abajo para probar que funciona antes de conectar con n8n.

In [ ]:
# CELDA 6 — Test del servidor (opcional)
import requests

response = requests.post(
    f'{tunnel.public_url}/tts',
    json={
        'text': 'Sin rollos, aquí va la verdad. La mayoría de la gente lleva años entrenando mal y nadie se lo ha explicado bien.',
        'language': 'es'
    }
)

if response.status_code == 200:
    with open('/content/test_juan_fit.wav', 'wb') as f:
        f.write(response.content)
    print('Test exitoso. Descarga /content/test_juan_fit.wav para escuchar la voz.')
else:
    print('Error:', response.text)